# HW1: Image Classification with Logistic Regression

CSCI 3320 · AI Programming in Python · Fall 2026

You have your own pair of image classes (the instructor assigned it, and the
[HW1 page of the quiz app](https://csci3320-quiz.azurewebsites.net/hw1) shows it). In this homework you teach a
**logistic regression** model to tell your two classes apart, using nothing but
the raw pixels, and you find out how far that simple model can go.

**Due: Thursday, October 22, 9:00 PM (Central Time).** The full instructions and
the grading are on the [assignment page](https://dongchul.kim/csci3320-fall2026/hw/1).

**What you submit**

1. `model.npz`, your final model, uploaded on the [HW1 page](https://csci3320-quiz.azurewebsites.net/hw1). It is graded
   at once on hidden test images. **Your grade uses the test accuracy of your LAST
   upload**, so make sure the last model you upload is your best one.
2. This notebook as one **`.ipynb` file** (File → Download → Download .ipynb), run from
   top to bottom with all outputs and your answers, uploaded to Brightspace. Not a PDF,
   not a Colab link.

**Rules.** Logistic regression only (no neural networks, no other models), on the
raw pixels of the images you were given (no other data, no pretrained features).
Work on your own. Every place you have to write code is marked **TODO**; every
question you have to answer is marked **Q**.

> In Colab, the default CPU runtime is enough. Run the cells in order.

## Step 0: Imports

In [ ]:
import os, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__)

## Step 1: Get your data

Two ways, pick one:

- **Paste your link (easiest).** On the [HW1 page](https://csci3320-quiz.azurewebsites.net/hw1), press **Copy link** and paste it
  between the quotes below. Colab downloads the file itself (about 45 MB, a few seconds).
- **Upload the file.** If you already downloaded `hw1_<your pair>.npz`, open the **Files** panel
  (the folder icon on the left), press the upload icon, and choose that file. The cell finds it by
  itself. Uploaded files disappear when Colab disconnects, so you would upload it again.

Your link and your file are personal: they hold your own pair.

In [ ]:
DATA_URL = "PASTE_YOUR_LINK_HERE"
FILE = "hw1_data.npz"

uploaded = sorted(f for f in os.listdir(".") if f.startswith("hw1_") and f.endswith(".npz"))
if not os.path.exists(FILE) and uploaded:
    FILE = uploaded[0]                      # a file you uploaded in the Files panel
if not os.path.exists(FILE):
    if not DATA_URL.startswith("http"):
        raise ValueError("Paste your link from the HW1 page into DATA_URL, "
                         "or upload your hw1_....npz file in the Files panel on the left.")
    print("downloading ...")
    urllib.request.urlretrieve(DATA_URL, FILE)
print("using", FILE)

d = np.load(FILE)
X, y, CLASSES = d["train_images"], d["train_labels"], [str(c) for c in d["classes"]]
print("images", X.shape, X.dtype, "| labels", y.shape, "| classes", CLASSES)
print("images per class:", np.bincount(y))

**Check:** `images (4000, 64, 64, 3) uint8 | labels (4000,) | classes [your two classes]` and 2,000 images per class. Label 0 is the first class, label 1 the second.

## Step 2: Look at your data

Never train on data you have not looked at. The first two rows are class 0, the
last two class 1. Below them is the **average image** of each class: every pixel
averaged over its 2,000 images.

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(4, 8, figsize=(14, 7.5))
for r in range(4):
    k = 0 if r < 2 else 1
    for c, i in enumerate(rng.choice(np.flatnonzero(y == k), 8, replace=False)):
        axes[r, c].imshow(X[i]); axes[r, c].axis("off")
    axes[r, 0].set_title(CLASSES[k], fontsize=10, loc="left")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(5, 2.6))
for k in (0, 1):
    axes[k].imshow(X[y == k].mean(axis=0).astype(np.uint8)); axes[k].set_title(f"average {CLASSES[k]}"); axes[k].axis("off")
plt.tight_layout(); plt.show()

**Q1.** Look at the two average images. What differs between them that a model working
on raw pixels could pick up (position, colour, brightness, shape)? What makes your two
classes hard to tell apart? (2 to 3 sentences)

*Your answer:*

## Step 3: Pixels in, a validation split, standardization

Logistic regression needs one row of numbers per image, so each 64 x 64 x 3 image
becomes **12,288 numbers**. Hold out 20% of the images to measure yourself, and
standardize every pixel with the mean and standard deviation of the **training**
images only (the validation images must not leak into it).

In [ ]:
Xf = X.reshape(len(X), -1).astype(np.float32)          # (4000, 12288), values 0..255
X_tr, X_va, y_tr, y_va = train_test_split(Xf, y, test_size=0.2, stratify=y, random_state=0)

scaler = StandardScaler().fit(X_tr)
Z_tr, Z_va = scaler.transform(X_tr), scaler.transform(X_va)
print("train", Z_tr.shape, "validation", Z_va.shape)

## Part A: A baseline with sklearn

Start with scikit-learn's `LogisticRegression` and its default settings, and look at
**both** accuracies: on the images it learned from, and on the validation images.

In [ ]:
t0 = time.time()
base = LogisticRegression(max_iter=2000).fit(Z_tr, y_tr)
print(f"default C={base.C}: train accuracy {base.score(Z_tr, y_tr):.3f}, "
      f"validation accuracy {base.score(Z_va, y_va):.3f}  ({time.time() - t0:.0f}s)")

**Q2.** How far apart are the training and the validation accuracy? What does that gap
tell you about the model? (2 sentences)

*Your answer:*

## Part B: Logistic regression from scratch

Now write the same model yourself, with the gradient descent loop from class. With
$z = Xw + b$ and $p = \sigma(z) = 1 / (1 + e^{-z})$, the average log loss is

$$L = \frac{1}{n} \sum_i \big[\log(1 + e^{z_i}) - y_i z_i\big],$$

and its gradients are

$$\frac{\partial L}{\partial w} = \frac{1}{n} X^\top (p - y), \qquad \frac{\partial L}{\partial b} = \frac{1}{n} \sum_i (p_i - y_i).$$

Fill in the **TODO**s: compute $p$, the two gradients, and the update. Everything is a
matrix operation; do not loop over the images.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))


def train_gd(Z, y, lr=0.05, iters=300):
    y = y.astype(Z.dtype)      # same number type as Z: mixing types makes numpy copy all of Z every step
    w = np.zeros(Z.shape[1], dtype=Z.dtype); b = 0.0; losses = []
    for t in range(iters):
        z = Z @ w + b
        losses.append(np.mean(np.logaddexp(0, z) - y * z))
        p = None        # TODO: predicted probabilities
        grad_w = None   # TODO: gradient of the loss with respect to w
        grad_b = None   # TODO: gradient of the loss with respect to b
        # TODO: update w and b with learning rate lr
    return w, b, losses


w, b, losses = train_gd(Z_tr, y_tr)
pred_va = ((Z_va @ w + b) > 0).astype(int)
print(f"from scratch: validation accuracy {(pred_va == y_va).mean():.3f}")
plt.plot(losses); plt.xlabel("iteration"); plt.ylabel("training loss"); plt.title("gradient descent"); plt.show()

**Check:** the loss falls smoothly, and the validation accuracy is in the same range as
Part A.

**Q3.** Try two other learning rates (one much larger, one much smaller) and describe in
2 sentences what happens to the loss curve. Then put `lr` back.

*Your answer:*

## Part C: Regularization: choosing C

`LogisticRegression` has a regularization strength `C`: a **small C** keeps the weights
small (a simpler model), a **large C** lets them grow. Fit one model for each C below,
record its training and validation accuracy, plot both against C, and keep the C with
the best **validation** accuracy. (The larger values of C take longer to fit.)

In [ ]:
Cs = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
train_acc, val_acc = [], []
for C in Cs:
    t0 = time.time()
    # TODO: fit LogisticRegression(C=C, max_iter=2000) on (Z_tr, y_tr) and append its
    #       training accuracy to train_acc and its validation accuracy to val_acc
    print(f"C={C:g}  ({time.time() - t0:.0f}s)")

best_C = None   # TODO: the C with the highest validation accuracy

plt.semilogx(Cs, train_acc, "o-", label="train"); plt.semilogx(Cs, val_acc, "o-", label="validation")
plt.xlabel("C"); plt.ylabel("accuracy"); plt.legend(); plt.grid(alpha=0.3); plt.show()
print("best C:", best_C, " validation accuracy:", max(val_acc))

**Q4.** Describe the two curves. Where does the model **underfit**, where does it
**overfit**, and why is your best C better than the default `C=1`? (3 sentences)

*Your answer:*

## Part D: Learning curve: does more data help?

Train with your best C on only the first `n` training images of each class, for several
`n`, and measure each model on the **same** validation set.

In [ ]:
sizes = [50, 100, 250, 500, 1000, 1600]
lc_train, lc_val = [], []
for n in sizes:
    idx = np.r_[np.flatnonzero(y_tr == 0)[:n], np.flatnonzero(y_tr == 1)[:n]]
    # TODO: fit LogisticRegression(C=best_C, max_iter=2000) on Z_tr[idx], y_tr[idx] and append
    #       its accuracy on those training images to lc_train and on the validation set to lc_val

plt.plot(sizes, lc_train, "o-", label="train"); plt.plot(sizes, lc_val, "o-", label="validation")
plt.xlabel("training images per class"); plt.ylabel("accuracy"); plt.legend(); plt.grid(alpha=0.3); plt.show()

**Q5.** What happens to the two curves as `n` grows? Would 10 times more images help your
model a lot, a little, or not at all? Explain from the plot. (2 to 3 sentences)

*Your answer:*

## Part E: Look at the mistakes

A single accuracy hides a lot. These are validation images your best model got wrong.

In [ ]:
model = LogisticRegression(C=best_C, max_iter=2000).fit(Z_tr, y_tr)
pred = model.predict(Z_va)
wrong = np.flatnonzero(pred != y_va)
print(f"{len(wrong)} of {len(y_va)} validation images are wrong")
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, i in zip(axes.ravel(), wrong[:12]):
    ax.imshow(X_va[i].reshape(64, 64, 3).astype(np.uint8)); ax.axis("off")
    ax.set_title(f"true {CLASSES[y_va[i]]}\npredicted {CLASSES[pred[i]]}", fontsize=8)
plt.tight_layout(); plt.show()

**Q6.** What do the mistakes have in common? Name one thing about these images that a
model on raw pixels cannot handle well. (2 to 3 sentences)

*Your answer:*

## Part F: The final model, and submitting it

Retrain on **all 4,000 images** with your best C, and save the model as `model.npz`. The
grader takes each hidden test image, flattens its raw pixels into `x` (12,288 values from
0 to 255, in the same order as `X.reshape(len(X), -1)`), and predicts

    class 1  if  ((x - mean) / scale) . w + b > 0,   otherwise class 0

so the file must hold your weights `w`, intercept `b`, and the `mean` and `scale` of your
`StandardScaler`.

In [ ]:
final_scaler = StandardScaler().fit(Xf)
final = LogisticRegression(C=best_C, max_iter=2000).fit(final_scaler.transform(Xf), y)
np.savez("model.npz", w=final.coef_[0], b=final.intercept_[0],
         mean=final_scaler.mean_, scale=final_scaler.scale_)
print("saved model.npz", os.path.getsize("model.npz") // 1000, "KB")

Check the file the way the grader reads it: its predictions must match the model's.

In [ ]:
def predict_like_the_grader(path, images_uint8):
    m = np.load(path)
    x = images_uint8.reshape(len(images_uint8), -1).astype(np.float32)
    return (((x - m["mean"]) / m["scale"]) @ m["w"] + m["b"] > 0).astype(int)

agree = (predict_like_the_grader("model.npz", X) == final.predict(final_scaler.transform(Xf))).mean()
print(f"the file and the model agree on {agree:.4f} of the images (should be 1.0000)")

Now download `model.npz` (the cell below, or the Files panel on the left) and upload it on
the [HW1 page](https://csci3320-quiz.azurewebsites.net/hw1). You see your model's accuracy on the hidden test set (1,000
images that are not in your training data) right away. You may submit up to 20 times in
any 24 hours.

**Your grade uses the test accuracy of your LAST upload, not your best one.** If your last
upload scores lower than an earlier one, the lower score is graded. Before the deadline,
make sure the last model you upload is your best model.

Finally, **Runtime → Run all** once more so every output is fresh, then download this
notebook as an `.ipynb` file (File → Download → Download .ipynb) and upload that file to
Brightspace. Not a PDF, not a Colab link: we open and check the `.ipynb` itself.

In [ ]:
try:
    from google.colab import files
    files.download("model.npz")
except Exception:
    print("model.npz is at", os.path.abspath("model.npz"))